In [1]:
## requires 
# pip install snowflake-connector-python[pandas]
# pip install sqlalchemy

Run Cells Sequentially
Each cell is numbered in the order they should be run in

In [15]:
#imports
import snowflake.connector
import pandas as pd
from sqlalchemy import VARCHAR, false, table
import os
import json
from configparser import ConfigParser

In [28]:
#utilities
def df(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = pd.DataFrame(connCur.fetch_pandas_all())
    connCur.close()
    return df

def listJson(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = connCur.fetchall()
    connCur.close()
    return df

def getDDL(conn,sql):
    connCur = conn.cursor()
    result = connCur.execute(sql).fetchone()
    connCur.close()
    return result

def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.CSV'):
            filesList.append(path + "\\" + x)
    return filesList

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def writeConfigParser(path,parserFile):
    f = open(path, 'w') 
    parserFile.write(f)

In [29]:
#set vars
lzSchema = "SAPH_USR2"
odsSchema = "SAPH"
LZdb = "LZ_DEV"
ODSdb = "LZ_DEV"
overwriteDDLs = True
#snow file paths, please make sure you update this to the correct file name
snowLZ_path = "C:\\Repo\\EnterpriseCloud\\Snow\\K.27674.01__gmxbase_LZTables.sql" 
snowODS_path = "C:\\Repo\\EnterpriseCloud\\Snow\\K.27674.02__gmxbase_ODSTables.sql"
configFile = "C:\\Repo\\EnterpriseCloud\\Configuration\\FileUploader_GMX.json"
repo_path = "C:\\Repo\\EnterpriseCloud\\"
templatePath = repo_path + "Notebooks\\CSVTemplates"
typeDict = {'nvarchar':'NVARCHAR',
            'numeric':'FLOAT',
            'datetime':'TIMESTAMP',
            'varchar':'VARCHAR',
            'boolean':'BOOLEAN',
            'decimal':'FLOAT',
            'float':'FLOAT',
            'str':'NVARCHAR',
            'bit':'BOOLEAN'}

In [30]:
# create LZ tables from CDS view standard templates
filesList = getFileList(templatePath)
#print(filesList)

df_tables = pd.DataFrame()

# read csv template
for file in filesList:
    csvData = pd.read_csv(file,nrows=1)
    #print(csvData)
    df_tables = df_tables.append(csvData,ignore_index=True)

df_tables["TABLE_NAME"] = df_tables["CDS View"] #+ "_" +  df_tables["Source.Name"]
#print(df_tables)

if overwriteDDLs:
    ddlAction = """create or replace TABLE """ + lzSchema + "." + df_tables['TABLE_NAME']
else:
    ddlAction = """create TABLE """ + lzSchema + "." + df_tables['TABLE_NAME'] + """ if not exists"""
    
#generate DDL
df_tables['lz_ddl'] = ddlAction + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model SAPAPI """ + df_tables['TABLE_NAME'] + """ file.';"""

print(df_tables['lz_ddl'].values)
# print(df_tables.head())  

["create or replace TABLE SAPH_USR2.SAP_US_MMT_EV_EBEW (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI SAP_US_MMT_EV_EBEW file.';"
 "create or replace TABLE SAPH_USR2.SAP_US_MMT_EV_MBEW (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI SAP_US_MMT_EV_MBEW file.';"
 "create or replace TABLE SAPH_USR2.SAP_US_MMT_EV_MSSA (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI SAP_US_MMT_EV_MSSA file.';"
 "create or replace TABLE SAPH_USR2.SAP_US_MMT_EV_QBEW (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model SAPAPI SAP_US_MMT_EV_QBEW file.';"
 "create or replace TABLE SAPH_USR2.SAP_US_MMT_EV_RSEG (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='M

In [31]:
#write out LZ database table ddl commands to snow file
outstr = ""
outstr = "USE DATABASE <<LZ>>;\n"
for idx, row in df_tables.iterrows():
    outstr += row['lz_ddl'] + """
"""
# print(outstr)
print(snowLZ_path)

writeFile(snowLZ_path,outstr)

C:\Repo\EnterpriseCloud\Snow\K.27674.01__gmxbase_LZTables.sql


In [32]:
csv_path = repo_path + "Notebooks\\Sf_characters.csv"
df_characters = pd.read_csv(csv_path)
print(df_characters)

  OLD NEW
0   -   _
1   /   _


In [33]:
# create ODS tables from standard CDS view templates to ODS snow file

# read csv template
outstr = "USE DATABASE <<ODS>>;\n"
#print(filesList)
for file in filesList:
    columnList = ''
    csvData = pd.read_csv(file)

    #loop through characters df and replace any illegal characters
    for idx,row in df_characters.iterrows():
        csvData["Technical Field Name(ODS)"]  = csvData["Technical Field Name(ODS)"].str.replace(row['OLD'],row['NEW'])
    
    #print(csvData.head(1000))
    tableName = csvData.at[0,"CDS View"] #+ "_" + csvData.at[0,"Source.Name"]
    csvData = csvData.replace({'Type':typeDict})
    csvData["columnList"] = '"' + csvData["Technical Field Name(ODS)"] + '" ' + csvData["Type"] + ","
    
    for idx,row in csvData.iterrows():
        columnList += row["columnList"] + '\n'
    if overwriteDDLs:
        ddlAction = 'create or replace TABLE ' + odsSchema + '.' + tableName + ' (\n'
    else:
        ddlAction = 'create TABLE ' + odsSchema  + ' if not exists (\n'
    controlColumns = '_FILEDATE TIMESTAMP_NTZ(9),\n _FILENAME varchar(128),\n AUDIT_LOG_ID varchar(128),\n _ROW_HASH varchar(128)\n);\n\n'
    
    outstr +=  ddlAction + columnList + controlColumns
    #print(outstr)
    #break

#print(ddlString) ## too big to print, check in file
print(snowODS_path)
writeFile(snowODS_path,outstr)

C:\Repo\EnterpriseCloud\Snow\K.27674.02__gmxbase_ODSTables.sql


In [34]:
# create the fileupoader.config entries 
# (remove existing entries if you need to overwrite them in teh fileuploader.config file)
#read existing file
#print(df_tables)
configFileList = getFile(configFile)

# create new list, remove empty lines
newFileList = []
for l in configFileList: 
    if l != "":
        newFileList.append(l)
  
# remove last line
newFileList = newFileList[:-1]

# create a working copy of df_tables to build config
df_tables_config = df_tables

#check to see if entry exists in file uploader config, if it does drop it from the df
for idx, row in df_tables_config.iterrows():
    dfTableName = "USR2_SF_" + row['CDS View']  
    config_exists = False
    for tn in newFileList:
        if dfTableName in tn:
            config_exists = True
    if config_exists:
        print('CDS view already in file uploader config, dropping',dfTableName,idx)        
        df_tables_config.drop(idx)
            
#create the new config entries
confstr = ""

for l in newFileList:
    confstr += l

confstr = confstr.rstrip()
confstr = confstr.rstrip('\r\n')
confstr = confstr.rstrip()
for idx, row in df_tables_config.iterrows():
    confTableName = row['TABLE_NAME'] 
    confFileName = "SF_" + row['CDS View']
    confName = '"USR2_SF_' + row['CDS View']
    confstr += """,
    """ + confName + """\": {
        "SNOWFLAKE_SCHEMA_NAME": \"""" + lzSchema + """\",
        "SNOWFLAKE_DATABASE_NAME": "<<LZ>>",
        "SNOWFLAKE_WAREHOUSE_NAME": "Import_File_WH",
        "SNOWFLAKE_ROLE_NAME": "<<SYSADMIN>>",
        "SNOWFLAKE_TABLE_NAME": \"""" + confTableName + """\",
        "SNOWFLAKE_FILE_FORMAT_NAME": "CONTROL.POWERSHELL_JSON",
        "SNOWFLAKE_STAGE_TYPE": "NAMED",
        "SNOWFLAKE_STAGE_NAME": "SAPH_USR2_STG",
        "FILE": "F:\\\\Shares\\\\Files\\\\GMX\\\\USR2\\\\""" + confFileName + """_*.json",
        "SPLIT_FILE": false
    }"""
    
# add final bracket
confstr += """
}"""

# write out new config file
#print(confstr)
writeFile(configFile,confstr)

In [35]:
#**************************** LZ to ODS dbt model Utility**********************
#helper functions for DBT model

def getJinjaConfig(strategy,cdsView,unique_keys):

	jinjaConfig = """
	{{ config(
	  materialized='incremental',
	  incremental_strategy='"""+ strategy +"""',
	  alias='""" + cdsView + """',
	  unique_key = ['"""+ unique_keys +"""'],
	  transient=false
	) }} """

	return jinjaConfig

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig,ifMerge):

	DBT_CODE = """

	/************
	Model file generated from gmxDbtTemplateUtility.ipynb 
	*************/

	"""+ jconfig +"""

	with source_data as (
	SELECT
		""" +col_s_string+ """,
		_FILEDATE,
		_FILENAME
	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}" """
	"""+ ifMerge +"""
	),
	
	transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
	from source_data
	),
	
	type_data as (
	SELECT
		"""+col_type_data_str+""",
		cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
		cast(_FILENAME as varchar(128)) as _FILENAME,
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH
	from transform_data
	)

	select * from type_data

			"""
	#print(DBT_CODE)
	return DBT_CODE

#DBT CODE FOR INSERT-OVERWRITES
def getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig):

	DBT_CODE_I = """

	/************
	Model file generated from gmxDbtTemplateUtility.ipynb 
	*************/

	"""+ jconfig +"""

    with max_f as (
		SELECT MAX(_FILENAME) as _filename 
		from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}"  """
	),

    max_d as (
		select s._filename, max(s._filedate) as _filedate from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}"  """ as s
		join max_f on s._filename = max_f._filename
		group by s._filename
	),

	source_data as (
	SELECT
		""" +col_s_string+ """,
		s._FILEDATE,
		s._FILENAME
	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }} s"  """
    join max_d on s._filedate = max_d._filedate and s._filename = max_d._filename
	
	),
	
	transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
	from source_data
	),
	
	type_data as (
	SELECT
		"""+col_type_data_str+""",
		cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
		cast(_FILENAME as varchar(128)) as _FILENAME,
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH
	from transform_data
	)

	select * from type_data

			"""
	#print(DBT_CODE)
	return DBT_CODE_I

#CREATE SCHEMA FILE
def createSchema(df_tables_test,lzSchema,odsSchema,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + lzSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename)
        writeFile(path_write,final_schema)
		

In [49]:
# Build dbt models from cds view templates


filesList = getFileList(templatePath)
#print(filesList)

# read csv template file and build list of required inputs
for csvFile in filesList:
    dashChar = csvFile.rfind("-")
    dotChar = csvFile.rfind(".")
    dbt_strategy = csvFile[dashChar+1:dotChar]
    columnList = ""
    csvDataOut = pd.DataFrame()
    csvData = pd.read_csv(csvFile)
    
    #print(csvData.head(5))
    cdsViewName = csvData.at[0,"CDS View"]
    tableName = csvData.at[0,"CDS View"] #+ "_" + csvData.at[0,"Source.Name"]
    baseTable = csvData.at[0,"Source.Name"]
    csvData = csvData.replace({'Type':typeDict})
    csvData["columnList"] = csvData["Technical Field Name(ODS)"] + " " + csvData["Type"] + ","
    dfKeys = csvData.query('PK=="yes"')["Technical Field Name(ODS)"]
    unique_fields=[]
    unique_fields_qualify = []
    unique_fields = dfKeys.values.tolist()
    
    unique_fields_str = "','".join(unique_fields)    
    for idx,row in df_characters.iterrows():
        csvData["ScrubbedColumnName"]  = csvData["Technical Field Name(ODS)"].str.replace(row['OLD'],row['NEW'])
    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_source = []
    col_type_data = []
    #type data columns
    for idx,row in csvData.iterrows():
        col_type_data.append('CAST("' + row["ScrubbedColumnName"] + '" AS ' + row["Type"] + ') AS "' + row["ScrubbedColumnName"] + '"')
        col_t.append(row["ScrubbedColumnName"])
        col_s.append('JSONFIELD:"' + row["Technical Field Name(ODS)"] + '"::STRING AS "' + row["ScrubbedColumnName"] + '"')  

    col_type_data_str = ', \n \t \t'.join(col_type_data)
    col_type_data_str = col_type_data_str.upper()
    #print(col_type_data_str)

    # column string for transform cte
    col_t_string = '"' + '", \n \t \t"'.join(col_t) + '"'
    col_t_string = col_t_string.upper()
    #print(col_t_string)
                 
    col_s_string = ', \n \t \t'.join(col_s)
    dbt_strategy = dbt_strategy.strip()
    if dbt_strategy == 'incremental':
        dbt_strategy = 'merge'
        
        unique_fields_qualify = ['JSONFIELD:"{0}"::STRING'.format(k) for k in unique_fields]
        unique_fields_qualify_str = ",".join(unique_fields_qualify)
        
        ifMerge = """{% if is_incremental() %}WHERE  _FILENAME > IFNULL((SELECT MAX(_FILENAME) FROM {{ this }}),'1970-01-01'){% endif %}
        QUALIFY (ROW_NUMBER() OVER (PARTITION BY """+unique_fields_qualify_str+"""  ORDER BY _FILENAME DESC,_FILEDATE DESC )) = 1"""  

    else:
        dbt_strategy = 'insert_overwrite'
        ifMerge = """WHERE REGEXP_SUBSTR(_FILENAME,'\\\d{14}') = (SELECT MAX(REGEXP_SUBSTR(_filename,'\\\d{14}')) from {{ source('SAPH', """ +"'"+tableName+ "'" """) }})"""

        unique_fields_str = ''
    
    #create dbt model jinja config
    jinjaConfig = getJinjaConfig(dbt_strategy,tableName,unique_fields_str)
    #print(jinjaConfig)
    dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig,ifMerge)
    #Creating code for dbt model
    # if dbt_strategy == 'merge':
    #     dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig,ifMerge)
    # else:
    #     dbt_model = getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tableName,odsSchema,jinjaConfig)
    
    #create dbt file
    dbt_path = repo_path + "dbt\\lz_to_ods\\models\\saph\\"
    filename = tableName + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)
    

In [37]:
#append Schema.yml file
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\saph\\"
df_tables_test = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#look for schema names
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path)
    for line in schema_file:        
        if 'identifier:' in line:
            cdsName = line.replace("identifier:","")
            cdsName = cdsName.replace("'","")
            cdsName = cdsName.replace(",","")
            cdsName = cdsName.strip()
            existing_schemas.append(cdsName)
#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
df_tables_test = pd.concat([df_tables_test,es],ignore_index=True)
createSchema(df_tables_test,lzSchema,odsSchema,dbt_path)

In [ ]:
## create pub ODS_ views
%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "ODS" "SAPH" "C:\Repo\EnterpriseCloud\dbt\ods_to_pub\models" "varun.x.tiwari@us.tel.com"


In [51]:
def createSchema_pub(df_tables_test,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] +""""""

        schema_str = """version: 2\nsources:\n  - name: ODS_SAPH\n    description: Landing Zone - SAPH Schema\n    database: "{{ env_var('SNOWFLAKE_ODS') }}"\n    schema: SAPH\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """"""

        # schema_str_t ="""\nmodels:"""
        
        # for idx, r in df_tables_test.iterrows():
        #     schema_str_t += r['models'] + """
        # """
        final_schema = schema_str
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename)
        writeFile(path_write,final_schema)
        

In [52]:
dbt_path = repo_path + "dbt\\ods_to_pub\\models\\saph\\"
df_tables_test = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#look for schema names
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path)
    for line in schema_file:        
        if 'identifier:' in line:
            cdsName = line.replace("identifier:","")
            cdsName = cdsName.replace("'","")
            cdsName = cdsName.replace(",","")
            cdsName = cdsName.strip()
            existing_schemas.append(cdsName)
#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
df_tables_test = pd.concat([df_tables_test,es],ignore_index=True)
createSchema_pub(df_tables_test,dbt_path)